In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
import scipy.stats as sci

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
data=pd.read_csv(r"/kaggle/input/manufacturing-dataset/y1AQEIpMTR2j7xgr9MH0_Manufacturing Dataset.csv")

In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
data.dtypes

# Changing Data Types

## Objects to Category

In [ ]:
data.select_dtypes(include='object').columns

In [ ]:
data['Production ID']=data['Production ID'].astype('category')

In [ ]:
data['Product Type']=data['Product Type'].astype('category')

In [ ]:
data['Shift']=data['Shift'].astype('category')

## Numeric to Objects

In [ ]:
data['Machine ID']=data['Machine ID'].astype('category')

In [ ]:
data.select_dtypes(include='category').columns

## Objects to Numeric

In [ ]:
data.select_dtypes(include='object').columns

In [ ]:
data.Date=pd.to_datetime(data.Date)

Adding a month column as we have to find month on month trend

In [ ]:
import datetime

In [ ]:
data['month']=data.Date.apply(lambda x:x.month_name())
data['month']=data['month'].astype('category')

In [ ]:
data.columns

In [ ]:
data.dtypes

# EDA

## 1. Dealing with IQR using outlier

- Calculate the Interquartile Range (IQR) for all the numerical columns and use the IQR to identify any potential outliers in these data points.

In [ ]:
col=data.select_dtypes(exclude=['category','datetime64[ns]'])
col.columns

In [ ]:
plt.figure(figsize=(6,6*len(col.columns)))
for i,j in enumerate(col.columns):
    plt.subplot(len(col.columns),1,i+1)
    sns.boxplot(data[j])
    p25,p75=np.percentile(data[j].dropna(),25),np.percentile(data[j].dropna(),75)
    IQR=p75-p25
    plt.title(f'25th percentile={round(p25,2)}\n75th percentile={round(p75,2)}\nIQR={round(IQR,2)}')
    
    

Only Defects and Production Time hours have outliers

## 2. Identify Missing Values Across Key Production Metrics:

Analyse the dataset to identify missing values across all the columns and calculate the total number of missing values for each of these columns. Describe your findings and then impute all the missing values with suitable data points.

In [ ]:
data.isna().sum()

**Defects, Maintenance Hours, Down time Hours and Rework Hours have missing values. We will check if there is some pattern in the data and if the missing values can be imputed, else we will delete the missing values**

***Imputing for Defects***

In [ ]:
ax=sns.histplot(data.Defects,bins=6,kde=True)
#for g in ax.axes.flat
ax.bar_label(ax.containers[0])
plt.xticks(np.arange(data.Defects.min(),data.Defects.max()+1,(data.Defects.max()-data.Defects.min())/6))
plt.show()

In [ ]:
data.Defects.mean(),data.Defects.median(),data.Defects.mode()

Based on the given values, defects can have any integral value >=0, so better to remove null values as imputing with any measure of central tendency can gives us a wrong picture

In [ ]:
data.dropna(subset='Defects',inplace=True)

In [ ]:
data.isna().sum()

***Imputing for Maintenance Hours***

In [ ]:
ax=sns.histplot(data['Maintenance Hours'],bins=5,kde=True)
#for g in ax.axes.flat
ax.bar_label(ax.containers[0])
plt.xticks(np.arange(data['Maintenance Hours'].min(),data['Maintenance Hours'].max()+1,(data['Maintenance Hours'].max()-data['Maintenance Hours'].min())/5))
plt.show()

In [ ]:
data['Maintenance Hours'].mean(),data['Maintenance Hours'].median(),data['Maintenance Hours'].mode()

Based on the given values, Maintenance Hours can have any value >0, so better to remove null values as imputing with any measure of central tendency can gives us a wrong picture

In [ ]:
data.dropna(subset='Maintenance Hours',inplace=True)
data.isna().sum()

***Imputing for Down time Hours***

In [ ]:
ax=sns.histplot(data['Down time Hours'],bins=6,kde=True)
#for g in ax.axes.flat
ax.bar_label(ax.containers[0])
plt.xticks(np.arange(data['Down time Hours'].min(),data['Down time Hours'].max()+1,(data['Down time Hours'].max()-data['Down time Hours'].min())/6))
plt.xlim(data['Down time Hours'].min(),data['Down time Hours'].max())
plt.show()

In [ ]:
data['Down time Hours'].mean(),data['Down time Hours'].median(),data['Down time Hours'].mode()

Based on the given values, Down time Hours can have any value >0, so better to remove null values as imputing with any measure of central tendency can gives us a wrong picture

In [ ]:
data.dropna(subset='Down time Hours',inplace=True)
data.isna().sum()

***Imputing for Rework Hours***

In [ ]:
ax=sns.histplot(data['Rework Hours'],bins=10,kde=True)
#for g in ax.axes.flat
ax.bar_label(ax.containers[0])
plt.xticks(np.arange(data['Rework Hours'].min(),data['Rework Hours'].max()+0.1,(data['Rework Hours'].max()-data['Rework Hours'].min())/10))
plt.show()

In [ ]:
data['Rework Hours'].mean(),data['Rework Hours'].median(),data['Rework Hours'].mode()

Based on the given values, Rework Hours can have any value >0, so better to remove null values as imputing with any measure of central tendency can gives us a wrong picture

In [ ]:
data.dropna(subset='Rework Hours',inplace=True)
data.isna().sum()

**All null values have been removed**

## 3. Relationship Between Costs:

- Is there a pattern between the cost of materials per unit and the hourly labor cost? Determine if higher costs in materials tend to coincide with higher labor costs.

In [ ]:
target=data[['Material Cost Per Unit','Labour Cost Per Hour']]
for i in ['spearman','pearson','kendall']:
    print(f"{i} correlation coefficient:\n {target.corr(i)} \n")


In [ ]:
sns.scatterplot(x=data['Material Cost Per Unit'],y=data['Labour Cost Per Hour'])

**Both the correlations and scatter plot that there is no relationship between labour and material costs**

## 4. Efficiency Across Shifts:

- Do different work shifts (Day, Swing, Night) show differences in how long products take to make or how much energy they use? Compare these shifts to see if one is more efficient or uses less energy.

In [ ]:
ax1=sns.countplot(x=data.Shift,order=data.Shift.value_counts(ascending=True).index)
ax1.bar_label(ax1.containers[0])

In [ ]:
a=data.groupby('Shift')['Energy Consumption kWh'].mean()
aa=pd.DataFrame(a)
aa.sort_values('Energy Consumption kWh')

ax2=sns.barplot(width=0.8,y=data['Energy Consumption kWh'],x=data.Shift,errorbar=None,order=aa.sort_values('Energy Consumption kWh').index)
ax2.bar_label(ax2.containers[0])
plt.show()

In [ ]:
sci.ttest_ind(
    a=data.loc[(data.Shift=='Swing'),'Energy Consumption kWh'],
    b=data.loc[data.Shift=='Day','Energy Consumption kWh'],
    alternative='two-sided')

In [ ]:
sci.ttest_ind(
    a=data.loc[(data.Shift=='Night'),'Energy Consumption kWh'],
    b=data.loc[data.Shift=='Day','Energy Consumption kWh'],
    alternative='two-sided')

In [ ]:
sci.ttest_ind(
    a=data.loc[(data.Shift=='Swing'),'Energy Consumption kWh'],
    b=data.loc[data.Shift=='Night','Energy Consumption kWh'],
    alternative='two-sided')

**From the graph we can see that there is increase in Energy Consumption from Swing to Day to Night shift, Night and Swing shift also have statistically significant difference**

In [ ]:
a=data.groupby('Shift')['Production Time Hours'].mean()
aa=pd.DataFrame(a)
aa.sort_values('Production Time Hours')
ax=sns.barplot(y=data['Production Time Hours'],x=data.Shift,errorbar=None,
               order=aa.sort_values('Production Time Hours').index)
ax.bar_label(ax.containers[0],fmt="%0.2f")
plt.show()

In [ ]:
sci.ttest_ind(
    a=data.loc[(data.Shift=='Swing'),'Production Time Hours'],
    b=data.loc[data.Shift=='Day','Production Time Hours'],
    alternative='two-sided')

In [ ]:
sci.ttest_ind(
    a=data.loc[(data.Shift=='Swing'),'Production Time Hours'],
    b=data.loc[data.Shift=='Night','Production Time Hours'],
    alternative='two-sided')

In [ ]:
sci.ttest_ind(
    a=data.loc[(data.Shift=='Day'),'Production Time Hours'],
    b=data.loc[data.Shift=='Night','Production Time Hours'],
    alternative='two-sided')

**From the graph we can see that there is increase in Production Time from Night to Swing to Day shift, however no shift as statistically significant difference in average time** 

**Pattern of Shift and Time and Pattern of Shift and Energy is different from each other** 

## 5. Monthly Production Trends:

- How does the average number of units produced change from month to month? Look for any patterns, such as times of the year when production increases or decreases significantly.

In [ ]:
plt.figure(figsize=(16,6))
ax=sns.lineplot(y=data['Units Produced'],x=data.Date)


In [ ]:
plt.figure(figsize=(16,6))
ax=sns.lineplot(y=data['Units Produced'],x=data.month,marker='o')


**December,July,March,May,November and September has the highest Units produced, October and January has lowest**

## 6. Variability in Production by Product Type:

- Which type of product shows the most variation in how much is produced? Measure this using standard deviation to find out which product type's production volume varies the most.

In [ ]:
stdev=data.groupby('Product Type')['Production Volume Cubic Meters'].std()


In [ ]:
plt.figure(figsize=(9,4))
ax=sns.scatterplot(x=stdev.index,y=stdev.values,hue=stdev.index)
for i in range(len(stdev)):
    plt.annotate(round(stdev[i],3),(stdev.index[i],stdev[i]-0.002),size=8)


**Appliance have the most variability, though no major difference between the variability**

## 7. The Role of Operator Count in Efficiency:

- How does the number of operators affect how many units are produced per hour? Check if having more operators leads to more efficient production.

In [ ]:
plt.figure(figsize=(8.9,5))
data['Production per hour']=data['Units Produced']/data['Production Time Hours']
opc=data.groupby('Operator Count')['Production per hour'].mean()
ope=pd.DataFrame(opc)
sns.scatterplot(y=ope['Production per hour'],x=ope.index)
plt.ylim(17,18.66)
plt.yticks(np.arange(17,18.66,0.33))
for i in range(len(ope)):
    plt.annotate(round(float(ope.values[i]),2),((ope.index[i]-0.1),ope.iloc[i]-0.1),size=8) 
plt.xticks(np.arange(1,5,1))
plt.show() 

**No impact on production volume of operator count**

## 8. Identifying the Machine with Most Defects:

- Which machine tends to produce the most defects, considering the total units it produces? Calculate the defect rate as defects per 100 units to make comparisons easier.

In [ ]:
data['Defect Rate per 100 units produced']=data['Defects']*100/data['Units Produced']

In [ ]:
plt.figure(figsize=(12,5))
mdef=data.groupby('Machine ID')['Defect Rate per 100 units produced'].mean()
mdefr=pd.DataFrame(mdef)
ax=sns.barplot(x=data['Machine ID'],y=data['Defect Rate per 100 units produced'],errorbar=None,
         order=mdefr.sort_values(by='Defect Rate per 100 units produced').index)
ax.bar_label(ax.containers[0],fmt="%0.3f")
plt.show()

**Machine 7,11,2 have the lowest Defect Rate, whereas Machine 19,15,16 have the highest defect rate**

## 9. How Environment Affects Scrap Rate:

- Do changes in temperature and humidity affect how much scrap (waste) is produced? Analyze the data to see if there's a correlation between environmental conditions and scrap rate.

In [ ]:
numer=['Scrap Rate','Average Temperature C','Average Humidity Percent'] 
g=sns.heatmap(data[numer].corr(),annot=True,linewidth=0.2)
g.set_xticklabels(g.get_xticklabels(), rotation = 45, fontsize = 8)
sns.pairplot(data[numer])

**No impact of scrap rate on enviornmental factors**